# SegFormer TS-Bank Audit Follow-Up

This notebook summarizes the `SegFormer-B2` audit package at `results/hard_negative_audit/segformer_b2_tsbank_round1` and answers three practical questions:

1. Which mined banks look most keepable after manual review?
2. Did audit-derived curated reruns beat the raw mined `thr080_mean082` winner?
3. Which representative `hard_fp / ambiguous / noise` cards are worth showing in slides or the paper?

In [ ]:
from __future__ import annotations

import csv
import html
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Image as DisplayImage, Markdown, display

WORKDIR = Path.cwd().resolve()
if (WORKDIR / 'results').exists() and (WORKDIR / 'notebooks').exists():
    ROOT = WORKDIR
elif WORKDIR.name == 'notebooks' and (WORKDIR.parent / 'results').exists():
    ROOT = WORKDIR.parent
else:
    ROOT = Path('..').resolve()

AUDIT_DIR = ROOT / 'results' / 'hard_negative_audit' / 'segformer_b2_tsbank_round1'
AUDIT_CSV = AUDIT_DIR / 'audit_samples.csv'
BANK_OVERVIEW_CSV = AUDIT_DIR / 'bank_overview.csv'
EXPERIMENTS_CSV = ROOT / 'results' / 'experiments.csv'

LAYER1_ORDER = ['hard_fp', 'ambiguous', 'noise', 'bad_crop', 'gt_issue']
LAYER2_ORDER = [
    'pavement_edge',
    'shadow_dark_stripe',
    'line_like_texture',
    'surface_boundary',
    'debris_object',
    'other_target_clutter',
]
LAYER1_COLORS = {
    'hard_fp': '#2a9d8f',
    'ambiguous': '#e9c46a',
    'noise': '#b0b7c3',
    'bad_crop': '#f4a261',
    'gt_issue': '#e76f51',
}

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

In [ ]:
def read_csv(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def load_latest_experiment_rows(path: Path) -> dict[str, dict]:
    latest_by_triplet: dict[tuple[str, str, str], dict] = {}
    with path.open('r', encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            key = (row['experiment_name'], row.get('stage', ''), row.get('split', ''))
            ts = row.get('timestamp_utc') or ''
            prev = latest_by_triplet.get(key)
            if prev is None or ts > (prev.get('timestamp_utc') or ''):
                latest_by_triplet[key] = row

    latest_by_name: dict[str, dict] = {}
    for row in latest_by_triplet.values():
        name = row['experiment_name']
        ts = row.get('timestamp_utc') or ''
        prev = latest_by_name.get(name)
        if prev is None or ts > (prev.get('timestamp_utc') or ''):
            latest_by_name[name] = row
    return latest_by_name


def to_float(value: str | None) -> float:
    return float(value) if value not in (None, '') else float('nan')


def display_table(rows: list[dict], columns: list[str]) -> None:
    header = ''.join(
        f'<th style="border:1px solid #ccc;padding:6px 8px;text-align:left;">{html.escape(col)}</th>'
        for col in columns
    )
    body = []
    for row in rows:
        cells = ''.join(
            f'<td style="border:1px solid #ccc;padding:6px 8px;">{html.escape(str(row.get(col, "")))}</td>'
            for col in columns
        )
        body.append(f'<tr>{cells}</tr>')
    table = (
        '<table style="border-collapse:collapse;">'
        f'<thead><tr>{header}</tr></thead>'
        f'<tbody>{"".join(body)}</tbody>'
        '</table>'
    )
    display(HTML(table))


audit_rows = read_csv(AUDIT_CSV)
bank_overview_rows = read_csv(BANK_OVERVIEW_CSV)
experiments_by_name = load_latest_experiment_rows(EXPERIMENTS_CSV)

print(f'Loaded {len(audit_rows)} audited rows across {len(bank_overview_rows)} banks.')
print(f'Audit CSV: {AUDIT_CSV.relative_to(ROOT)}')
print(f'Experiments CSV: {EXPERIMENTS_CSV.relative_to(ROOT)}')

## 1. Audit Composition

The first pass is to compare bank quality before touching training: which banks have the highest reviewed keepable share, and what kinds of nuisance patterns dominate those keepable crops?

In [ ]:
summary_rows = []
layer1_by_bank: dict[str, Counter] = {}
keepable_taxonomy_by_bank: dict[str, Counter] = {}

for bank_label in sorted({row['bank_label'] for row in audit_rows}):
    subset = [row for row in audit_rows if row['bank_label'] == bank_label]
    layer1 = Counter((row.get('layer1_review_label') or '').strip() for row in subset)
    keepable = [
        row for row in subset
        if (row.get('layer1_review_label') or '').strip() in {'hard_fp', 'ambiguous'}
    ]
    keep_l2 = Counter((row.get('layer2_review_taxonomy') or '').strip() for row in keepable)

    layer1_by_bank[bank_label] = layer1
    keepable_taxonomy_by_bank[bank_label] = keep_l2

    summary_rows.append(
        {
            'bank': bank_label,
            'total': len(subset),
            'hard_fp': layer1.get('hard_fp', 0),
            'ambiguous': layer1.get('ambiguous', 0),
            'noise': layer1.get('noise', 0),
            'bad_crop': layer1.get('bad_crop', 0),
            'gt_issue': layer1.get('gt_issue', 0),
            'keepable': len(keepable),
            'keepable_share_pct': f"{100.0 * len(keepable) / len(subset):.1f}",
            'top_keepable_taxonomy': ', '.join(
                f"{label}={count}" for label, count in keep_l2.most_common(3)
            ),
        }
    )

summary_rows

In [ ]:
display_table(
    summary_rows,
    [
        'bank',
        'total',
        'hard_fp',
        'ambiguous',
        'noise',
        'bad_crop',
        'gt_issue',
        'keepable',
        'keepable_share_pct',
        'top_keepable_taxonomy',
    ],
)

best_keepable = max(summary_rows, key=lambda row: float(row['keepable_share_pct']))
display(
    Markdown(
        f"**Highest keepable share:** `{best_keepable['bank']}` with "
        f"`{best_keepable['keepable']}/{best_keepable['total']} = {best_keepable['keepable_share_pct']}%`."
    )
)

In [ ]:
banks = [row['bank'] for row in summary_rows]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(banks))
bottom = np.zeros(len(banks))
for label in LAYER1_ORDER:
    values = [layer1_by_bank[bank].get(label, 0) for bank in banks]
    if not any(values):
        continue
    axes[0].bar(
        x,
        values,
        bottom=bottom,
        label=label,
        color=LAYER1_COLORS.get(label, '#4c72b0'),
    )
    bottom += np.array(values)
axes[0].set_xticks(x)
axes[0].set_xticklabels(banks, rotation=20, ha='right')
axes[0].set_ylabel('audited sample count')
axes[0].set_title('Layer-1 review distribution by bank')
axes[0].legend()

keepable_share = [float(row['keepable_share_pct']) for row in summary_rows]
axes[1].bar(banks, keepable_share, color='#4c72b0')
axes[1].set_ylim(0, max(40, max(keepable_share) + 5))
axes[1].set_ylabel('keepable share (%)')
axes[1].set_title('Keepable share (hard_fp + ambiguous)')
for idx, value in enumerate(keepable_share):
    axes[1].text(idx, value + 0.6, f'{value:.1f}%', ha='center', va='bottom')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 2. Raw Mined Winner vs Curated Reruns

This comparison answers the real decision question: did manual curation produce a bank that beats the current `SegFormer-B2` mined validation winner, or is the audit more useful as an explanatory tool?

In [ ]:
comparison_config = [
    {
        'label': 'raw mined winner',
        'experiment_name': 'segformer_b2_b1_tsbank_thr080_mean082',
        'bank': 'thr080_mean082',
        'filter': 'raw mined',
    },
    {
        'label': 'thr080_mean082 hard_fp',
        'experiment_name': 'segformer_b2_b1_thr080_mean082_curated_hardfp_seed042_uavval',
        'bank': 'thr080_mean082',
        'filter': 'hard_fp',
    },
    {
        'label': 'thr080_mean082 hard_fp + ambiguous',
        'experiment_name': 'segformer_b2_b1_thr080_mean082_curated_hfpa_seed042_uavval',
        'bank': 'thr080_mean082',
        'filter': 'hard_fp + ambiguous',
    },
    {
        'label': 'thr080_mean080 hard_fp + ambiguous',
        'experiment_name': 'segformer_b2_b1_thr080_mean080_curated_hfpa_seed042_uavval',
        'bank': 'thr080_mean080',
        'filter': 'hard_fp + ambiguous',
    },
    {
        'label': 'thr080_mean080 hard_fp',
        'experiment_name': 'segformer_b2_b1_thr080_mean080_curated_hardfp_seed042_uavval',
        'bank': 'thr080_mean080',
        'filter': 'hard_fp',
    },
]

comparison_rows = []
for item in comparison_config:
    row = experiments_by_name[item['experiment_name']]
    comparison_rows.append(
        {
            'label': item['label'],
            'bank': item['bank'],
            'filter': item['filter'],
            'experiment_name': item['experiment_name'],
            'IoU': f"{to_float(row['metric_iou']):.4f}",
            'F1': f"{to_float(row['metric_f1']):.4f}",
            'precision': f"{to_float(row['metric_precision']):.4f}",
            'recall': f"{to_float(row['metric_recall']):.4f}",
            'best_epoch': row.get('best_epoch', ''),
        }
    )

comparison_rows

In [ ]:
display_table(
    comparison_rows,
    ['label', 'bank', 'filter', 'IoU', 'F1', 'precision', 'recall', 'best_epoch'],
)

raw_winner = comparison_rows[0]
curated_only = comparison_rows[1:]
best_curated = max(curated_only, key=lambda row: float(row['IoU']))
mean082_hardfp = next(row for row in curated_only if row['label'] == 'thr080_mean082 hard_fp')
mean082_hfpa = next(row for row in curated_only if row['label'] == 'thr080_mean082 hard_fp + ambiguous')
mean080_hardfp = next(row for row in curated_only if row['label'] == 'thr080_mean080 hard_fp')
mean080_hfpa = next(row for row in curated_only if row['label'] == 'thr080_mean080 hard_fp + ambiguous')

raw_iou = float(raw_winner['IoU'])
best_curated_iou = float(best_curated['IoU'])
mean082_gap = float(mean082_hardfp['IoU']) - float(mean082_hfpa['IoU'])
mean080_gap = float(mean080_hardfp['IoU']) - float(mean080_hfpa['IoU'])

summary_md = f"""
**Best curated export:** `{best_curated['label']}` at `IoU {best_curated_iou:.4f}`.

**Delta vs raw mined winner:** `{best_curated_iou - raw_iou:+.4f}` IoU.

**Ambiguous effect on `thr080_mean082`:** `{mean082_gap:+.4f}` IoU when moving from `hard_fp + ambiguous` to `hard_fp` only.

**Ambiguous effect on `thr080_mean080`:** `{mean080_gap:+.4f}` IoU when moving from `hard_fp + ambiguous` to `hard_fp` only.
"""

display(Markdown(summary_md))

In [ ]:
labels = [row['label'] for row in comparison_rows]
iou_values = [float(row['IoU']) for row in comparison_rows]
f1_values = [float(row['F1']) for row in comparison_rows]
colors = ['#1d3557'] + ['#457b9d', '#8ecae6', '#457b9d', '#8ecae6']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(labels, iou_values, color=colors)
axes[0].set_ylabel('IoU')
axes[0].set_title('Raw mined winner vs curated reruns')
axes[0].tick_params(axis='x', rotation=25)
for idx, value in enumerate(iou_values):
    axes[0].text(idx, value + 0.003, f'{value:.4f}', ha='center', va='bottom', fontsize=9)

axes[1].bar(labels, f1_values, color=colors)
axes[1].set_ylabel('F1')
axes[1].set_title('F1 comparison')
axes[1].tick_params(axis='x', rotation=25)
for idx, value in enumerate(f1_values):
    axes[1].text(idx, value + 0.003, f'{value:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Representative Review Cards

Use the helper below to browse representative review cards by bank and layer-1 label. The default examples focus on the current `thr080_mean082` line because that bank remains the promoted `SegFormer-B2 B1` source.

In [ ]:
def show_examples(bank_label: str, layer1_label: str, limit: int = 3) -> None:
    subset = [
        row for row in audit_rows
        if row['bank_label'] == bank_label
        and (row.get('layer1_review_label') or '').strip() == layer1_label
    ]
    subset.sort(key=lambda row: int(row['rank_overall']))

    display(Markdown(f'### {bank_label} | {layer1_label} | {len(subset)} reviewed examples'))
    for row in subset[:limit]:
        card_path = AUDIT_DIR / row['review_card_rel']
        caption = (
            f"rank={row['rank_overall']}, score={row['score']}, "
            f"layer2={(row.get('layer2_review_taxonomy') or '<blank>')}"
        )
        display(Markdown(f'**{card_path.name}**  \\n{caption}'))
        display(DisplayImage(filename=str(card_path), width=900))


show_examples('thr080_mean082', 'hard_fp', limit=3)
show_examples('thr080_mean082', 'ambiguous', limit=3)
show_examples('thr080_mean082', 'noise', limit=3)